### Imports

In [2]:
import pandas as pd

### Load raw data

In [3]:
df = pd.read_csv("../data/raw/raw_walks.csv")

### Explore initial data
- Shape
- Basic info
- Basic statistics
- Check for null values

There are some null values in the elevation, temperature and humidity, but for the use case of this project that is okay

In [4]:
# Check the shape of the dataset
df.shape

(110, 18)

In [5]:
# Basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   workoutActivityType  110 non-null    object 
 1   duration             110 non-null    float64
 2   durationUnit         110 non-null    object 
 3   sourceName           110 non-null    object 
 4   sourceVersion        110 non-null    object 
 5   device               110 non-null    object 
 6   creationDate         110 non-null    object 
 7   startDate            110 non-null    object 
 8   endDate              110 non-null    object 
 9   elevation_ascended   82 non-null     object 
 10  humidity             104 non-null    object 
 11  temperature          104 non-null    object 
 12  distance_km          110 non-null    float64
 13  basal_calories       110 non-null    float64
 14  heart_rate_average   110 non-null    float64
 15  heart_rate_min       110 non-null    int

In [6]:
# Basic statistics for each column
df.describe()

,duration,distance_km,basal_calories,heart_rate_average,heart_rate_min,heart_rate_max,active_calories
count,110.000000,110.000000,110.000000,110.000000,110.000000,110.000000,110.000000
mean,38.614552,2.650604,63.734555,121.359938,98.572727,143.045455,191.970656
std,28.261749,1.859890,46.906484,13.262335,12.031655,16.064312,148.381788
min,1.101888,0.053492,0.946222,98.575000,74.000000,102.000000,2.482000
25%,19.693542,1.494712,32.475950,110.650500,90.250000,130.000000,100.595500
50%,30.692032,2.038630,50.450700,120.329500,98.000000,140.500000,146.654500
75%,49.060631,3.542542,81.054650,130.846750,107.750000,156.750000,247.098750
max,151.492048,11.585600,251.127000,150.684000,128.000000,180.000000,816.914000


In [7]:
# Check for null values in each column
df.isnull().sum()

workoutActivityType     0
duration                0
durationUnit            0
sourceName              0
sourceVersion           0
device                  0
creationDate            0
startDate               0
endDate                 0
elevation_ascended     28
humidity                6
temperature             6
distance_km             0
basal_calories          0
heart_rate_average      0
heart_rate_min          0
heart_rate_max          0
active_calories         0
dtype: int64

### Rename columns
Rename some columns for better understanding and organization

In [8]:
# View list of initial columns
df.columns

Index(['workoutActivityType', 'duration', 'durationUnit', 'sourceName',
       'sourceVersion', 'device', 'creationDate', 'startDate', 'endDate',
       'elevation_ascended', 'humidity', 'temperature', 'distance_km',
       'basal_calories', 'heart_rate_average', 'heart_rate_min',
       'heart_rate_max', 'active_calories'],
      dtype='object')

In [9]:
# Rename columns for clarity
df = df.rename(columns={
    "elevation_ascended": "elevation_ascended_cm",
    "basal_calories": "calories_basal",
    "active_calories": "calories_active",
})

### Create new columns
Create new columns that were not included in the orginal data export. 
- Pace and speed
- Ranges and totals
- Further date information

In [10]:
# Speed and pace
df["pace"] = df["duration"] / df["distance_km"]
df["speed"] = df["distance_km"] / (df["duration"] / 60.0)

# Range and totals
df["heart_rate_range"] = df["heart_rate_max"] - df["heart_rate_min"]
df["calories_total"] = df["calories_active"] + df["calories_basal"]
df["calories_per_km"] = df["calories_total"] / (df["distance_km"])

# Date information
df["year"] = pd.to_datetime(df["startDate"]).dt.year
df["month"] = pd.to_datetime(df["startDate"]).dt.month
df["day"] = pd.to_datetime(df["startDate"]).dt.day
df["day_of_week"] = pd.to_datetime(df["startDate"]).dt.dayofweek

# Season
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Fall"
df["season"] = df["month"].apply(season)

# Check new list of columns
df.columns

Index(['workoutActivityType', 'duration', 'durationUnit', 'sourceName',
       'sourceVersion', 'device', 'creationDate', 'startDate', 'endDate',
       'elevation_ascended_cm', 'humidity', 'temperature', 'distance_km',
       'calories_basal', 'heart_rate_average', 'heart_rate_min',
       'heart_rate_max', 'calories_active', 'pace', 'speed',
       'heart_rate_range', 'calories_total', 'calories_per_km', 'year',
       'month', 'day', 'day_of_week', 'season'],
      dtype='object')

### Drop unnecessary columns

In [11]:
# df = df.drop(columns=["startDate", "endDate", "sourceName", "sourceVersion", "device", "workoutActivityType"])

df = df.drop(columns=["workoutActivityType", "durationUnit", "sourceName", "sourceVersion", "device", "creationDate"], errors="ignore")

df.head(2)

,duration,startDate,endDate,elevation_ascended_cm,humidity,temperature,distance_km,calories_basal,heart_rate_average,heart_rate_min,...,pace,speed,heart_rate_range,calories_total,calories_per_km,year,month,day,day_of_week,season
0,23.467609,2024-03-09 17:03:18 -0300,2024-03-09 17:26:46 -0300,1565 cm,5500 %,43.142 degF,1.70326,38.2454,106.047,82,...,13.778054,4.354751,42,122.1538,71.717647,2024,3,9,5,Spring
1,45.893084,2024-03-16 14:36:43 -0300,2024-03-16 15:23:49 -0300,5159 cm,5100 %,47.642 degF,3.02969,76.0570,106.438,74,...,15.147782,3.960976,50,263.3920,86.936947,2024,3,16,5,Spring


In [14]:
df = df.drop(columns=["elevation_ascended_cm", "temperature", "humidity"], errors="ignore")

df.head(2)

,duration,startDate,endDate,distance_km,calories_basal,heart_rate_average,heart_rate_min,heart_rate_max,calories_active,pace,speed,heart_rate_range,calories_total,calories_per_km,year,month,day,day_of_week,season
0,23.467609,2024-03-09 17:03:18 -0300,2024-03-09 17:26:46 -0300,1.70326,38.2454,106.047,82,124,83.9084,13.778054,4.354751,42,122.1538,71.717647,2024,3,9,5,Spring
1,45.893084,2024-03-16 14:36:43 -0300,2024-03-16 15:23:49 -0300,3.02969,76.0570,106.438,74,124,187.3350,15.147782,3.960976,50,263.3920,86.936947,2024,3,16,5,Spring


### Convert column types
Converting columns to proper types

In [15]:
# Check types before conversion
print(df.dtypes)

duration              float64
startDate              object
endDate                object
distance_km           float64
calories_basal        float64
heart_rate_average    float64
heart_rate_min          int64
heart_rate_max          int64
calories_active       float64
pace                  float64
speed                 float64
heart_rate_range        int64
calories_total        float64
calories_per_km       float64
year                    int32
month                   int32
day                     int32
day_of_week             int32
season                 object
dtype: object


In [16]:
# Convert types
df["startDate"] = pd.to_datetime(df["startDate"], errors="coerce")
df["endDate"] = pd.to_datetime(df["endDate"], errors="coerce")

# Check types after conversion
print(df.dtypes)

duration                                float64
startDate             datetime64[ns, UTC-03:00]
endDate               datetime64[ns, UTC-03:00]
distance_km                             float64
calories_basal                          float64
heart_rate_average                      float64
heart_rate_min                            int64
heart_rate_max                            int64
calories_active                         float64
pace                                    float64
speed                                   float64
heart_rate_range                          int64
calories_total                          float64
calories_per_km                         float64
year                                      int32
month                                     int32
day                                       int32
day_of_week                               int32
season                                   object
dtype: object


### Drop outliner walks
Drop walks that aren't long enough to be considered, and pose risk to skewing the data 

In [17]:
# Drop walks under half KM, and walks under 10 mins
df = df[df["distance_km"] >= 0.5]
df = df[df["duration"] >= 10]

df.shape

(106, 19)

### Drop duplicates

In [18]:
# Check for duplicates and drop
df = df.drop_duplicates()

df.shape

(106, 19)

### Reorder columns

In [19]:
# Reorder columns

newOrder = [
    "startDate",
	"endDate",
	"distance_km",
	"duration",
	"pace",
	"speed",
	"calories_active",
	"calories_basal",
	"calories_total",
	"calories_per_km",
	"heart_rate_average",
	"heart_rate_min",
	"heart_rate_max",
	"heart_rate_range",
	"year",
	"season",
	"month",
	"day",
	"day_of_week"
]

df = df[newOrder]

df.tail(2)

,startDate,endDate,distance_km,duration,pace,speed,calories_active,calories_basal,calories_total,calories_per_km,heart_rate_average,heart_rate_min,heart_rate_max,heart_rate_range,year,season,month,day,day_of_week
108,2026-09-12 13:51:44-03:00,2026-09-12 14:13:14-03:00,2.01210,21.506536,10.688602,5.613456,81.5524,35.6172,117.1696,58.232493,105.031,82,122,40,2026,Fall,9,12,5
109,2026-09-26 11:46:35-03:00,2026-09-26 13:10:21-03:00,4.90754,83.770599,17.069774,3.514985,351.7080,138.7220,490.4300,99.933979,106.949,90,136,46,2026,Fall,9,26,5


### Export clean csv

In [20]:
# Export to csv
df.to_csv("../data/processed/clean_walks.csv", index=False)